Analysis of Ethereum consensus layer attestation propagation across the p2p network. Covers single attestations, aggregate attestations, and their relationship to block arrival timing, using telemetry from a distributed set of control nodes.

In [ ]:
# Imports
import polars as pl
import plotly.express as px
from IPython.display import display

from loaders import load_parquet
from utils import render_table

# Global Variables
target_date = None # Use this as a default for the automation and the rendering of the page

# one from the list ["save", "show"]
# default to "show" for the website rendering
render_method = "show"

def render_plot_fn(fig, method: str, path: str):
    if method == "image" and path != "":
        fig.write_image(path)
    elif method == "show":
        fig.show()
    else:
        raise f"method {method} not supported"
    

In [ ]:
# Load the raw datasets coming from the automated queries
# Single attestations
attestation_df = load_parquet("attestation_arrivals", target_date)
attestation_df = pl.from_pandas(attestation_df)

# Aggregations
pd_df = load_parquet("aggregation_broadcast_info", target_date)
aggregations_df = pl.from_pandas(pd_df)

# Blocks and data-columns
block_and_column_df = load_parquet("block_and_column_broadcast_info", target_date)
block_and_column_df = (
    pl.from_pandas(block_and_column_df)
    .rename({"bs.slot": "slot"})
)


## Attestation Arrivals

Distribution of single attestations by the time they were first observed on the network, measured from slot start. Attestations are expected to be broadcast within the first 4 seconds of a slot. The following graph shows the boundaries for each of the slot duties with vertical lines at seconds 8 (purple) and 12 (red). The split between included and non-included attestations reveals how much of the attestation traffic never makes it into a block.

In [ ]:
# Histogram over the first arrival of the attestations

df = (
    attestation_df
    .with_columns(
        total_attestations = pl.lit(len(attestation_df)),
        tag = pl.when(
            pl.col("block_slot").gt(0)).then(pl.lit("included_att"))
            .otherwise(pl.lit("non_included_att"))
    )
    .group_by(["latency_bucket", "tag"])
    .agg(
        attestations = pl.col("slot").count(),
        percentage = pl.col("slot").count() * 100 / pl.col("total_attestations").max(),
    )
    .sort(["latency_bucket"])
)

# the histogram of when where the attestations seen
h = px.bar(
    df,
    x="latency_bucket",
    y="percentage",
    color="tag",
)
h.update_xaxes(range=[0, 12])
h.update_layout(
    title="Histogram of when attestations where seen for the first time",
    xaxis_title_text="Seconds since slot started",
    yaxis_title_text="% of Attestations",
    width=1200,
    height=800,
)
h.update_xaxes(range=[0, 18])
h.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
h.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
render_plot_fn(h, render_method, "./images/agg/att_histogram_by_first_seen_tot.png")

### Never included attestations 
Attestations that were seen on the p2p network, but that weren't included into a block.

In [ ]:
# render the never included attestations

render_table(
    attestation_df
    .filter(pl.col("block_slot").lt(1))
    .group_by(["com_idx"])
    .agg(attestations = pl.col("val_idx").count())
)

render_table(
    attestation_df
    .filter(pl.col("block_slot").lt(1))
    .select(["slot", "val_idx", "att_first_seen_wb", "att_broadcast_p50", "inclusion_delay"])
)


## Attestation first seen by inclusion delay

Same histogram as above, colored by how many slots later each attestation was eventually included in a block. This reveals whether propagation timing predicts inclusion quality.

In [ ]:
# same histogram, but using the inclusion time as legend
df_2 = (
    attestation_df
    .with_columns(
        total_attestations = pl.lit(len(attestation_df)),
    )
    .group_by(["latency_bucket", "inclusion_range"])
    .agg(
        attestations = pl.col("slot").count(),
        percentage = pl.col("slot").count() * 100 / pl.col("total_attestations").max(),
        inclusion= pl.col("inclusion_delay").mean(),
    )
    .sort("inclusion", "latency_bucket", "percentage", descending=True)
)

h_2 = px.bar(
    df_2,
    x="latency_bucket",
    y="percentage",
    color="inclusion_range",
    opacity=0.9,
)
h_2.update_layout(
    title="Histogram of when attestations where seen for the first time and their inclusion",
    xaxis_title_text="Seconds since slot started",
    yaxis_title_text="% of Attestations",
    barmode='stack',
    width=1200,
    height=800,
)
h_2.update_xaxes(range=[0, 14])
h_2.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
h_2.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
render_plot_fn(h, render_method, f"./images/agg/att_histogram_by_first_seen_by_inclusion_tot.png")


## Attestation Arrivals by Network Coverage

Attestation propagation from the perspective of the full control node set. Instead of first-seen, this uses the time at which p50 and p90 of control nodes had received each attestation.

In [ ]:
# Histogram over the first arrival of the attestations
def arrival_of_attestations_over_percentile_with_inclusion(att_df: pl.DataFrame, perc: str = 'p50'):
    df_2 = (
        att_df
        .with_columns(
            total_attestations = pl.lit(len(att_df)),
        )
        .group_by([f"broadcast_{perc}_bucket", "inclusion_range"])
        .agg(
            attestations = pl.col("slot").count(),
            percentage = pl.col("slot").count() * 100 / pl.col("total_attestations").max(),
            inclusion= pl.col("inclusion_delay").mean(),
            broadcast= pl.col(f"att_broadcast_{perc}").mean(),
        )
        .sort("inclusion", "broadcast", "percentage", descending=True)
    )
    h_2 = px.bar(
        df_2,
        x=f"broadcast_{perc}_bucket",
        y="percentage",
        color="inclusion_range",
        opacity=0.9,
    )
    h_2.update_layout(
        title=f"Histogram of when attestations where seen by {perc} and their inclusion",
        xaxis_title_text=f"Seconds",
        yaxis_title_text="% of Attestations",
        barmode='stack',
        width=1200,
        height=800,
    )
    h_2.update_xaxes(range=[0, 10])
    h_2.add_vline(x=4, line_width=3, line_dash="dash", line_color="purple")
    h_2.add_vline(x=8, line_width=3, line_dash="dash", line_color="red")
    render_plot_fn(h_2, render_method, f"./images/agg/att_histogram_by_{perc}_and_inclusion_tot.png")

arrival_of_attestations_over_percentile_with_inclusion(attestation_df, "p50")
arrival_of_attestations_over_percentile_with_inclusion(attestation_df, "p90")

## Attestation propagation spread (p50 / p90)

Distribution of attestation propagation times across the network, colored by inclusion delay. The p50 broadcast time captures when half the control nodes have seen the attestation, while p90 captures near-full network coverage. A wide tail to the right suggests a subset of attestations is slow to reach the broader network, potentially impacting aggregator quality.

In [ ]:
# Histogram over the first arrival of the attestations
def arrival_of_attestations_over_percentile(att_df: pl.DataFrame, perc: str = 'p50'):
    df = (
        att_df
        .with_columns(
            total_attestations = pl.lit(len(att_df)),
        )
        .group_by([f"broadcast_{perc}_bucket_wfs"])
        .agg(
            attestations = pl.col("slot").count(),
            percentage = pl.col("slot").count() * 100 / pl.col("total_attestations").max(),
            broadcast= pl.col(f"att_broadcast_{perc}").mean(),
        )
        .sort(["broadcast"], descending=False)
    )
    
    # the histogram of when where the attestations seen
    f = px.bar(
        df,
        x=f"broadcast_{perc}_bucket_wfs",
        y="percentage",
    )
    f.update_layout(
        title=f"Histogram of when attestations where seen by {perc} of control nodes",
        xaxis_title_text=f"{perc} propagation (s)",
        yaxis_title_text="% of Attestations",
        width=1200,
        height=800,
    )
    f.update_xaxes(range=[0, 14])
    f.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
    render_plot_fn(f, render_method, f"./images/agg/att_histogram_first_seen_by_{perc}_tot.png")
    
arrival_of_attestations_over_percentile(attestation_df, "p50")
arrival_of_attestations_over_percentile(attestation_df, "p90")

## Attestation network coverage without inclusion split

Plain view of p50 and p90 propagation times (no inclusion coloring). Useful for reading off the raw propagation percentile distribution: where does the bulk of the network see most attestations, and how much headroom exists before the 8s and 12s protocol deadlines.

In [ ]:
# same histogram, but using the inclusion time as legend
df_2 = (
    attestation_df
    .with_columns(
        total_attestations = pl.lit(len(attestation_df)),
    )
    .group_by(["broadcast_p50_bucket_wfs", "inclusion_range"])
    .agg(
        attestations = pl.col("slot").count(),
        percentage = pl.col("slot").count() * 100 / pl.col("total_attestations").max(),
        inclusion= pl.col("inclusion_delay").mean(),
        broadcast= pl.col("att_broadcast_p50").mean(),
    )
    .sort("inclusion", "broadcast", "percentage", descending=True)
)

h_2 = px.bar(
    df_2,
    x="broadcast_p50_bucket_wfs",
    y="percentage",
    color="inclusion_range",
    opacity=0.9,
)
h_2.update_layout(
    title="P50 histogram of the attestation propagation and their inclusion",
    xaxis_title_text="Seconds",
    yaxis_title_text="% of Attestations",
    barmode='stack',
    width=1200,
    height=800,
)
h_2.update_xaxes(range=[0, 12])
h_2.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
h_2.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
render_plot_fn(h_2, render_method, "./images/agg/att_histogram_by_p50_by_inclusion_tot.png")

## P50 propagation vs inclusion delay

P50 broadcast time histogram using the inclusion delay as legend. The graph shows whether attestations that take longer to spread across the network are also included in later blocks.

In [ ]:
# correlation between first seen and propagation by P50/P90/P95
def histogram_per_first_att_seen_and_broadcast_percentile(att_df: pl.DataFrame, percentile: str):
    df = (
        att_df
        .with_columns(
            total_attestations = pl.lit(len(att_df)),
        )
        .group_by(["latency_bucket", f"broadcast_{percentile}_bucket_g"])
        .agg(
            attestations = pl.col("slot").count(),
            percentage = pl.col("slot").count() * 100 / pl.col("total_attestations").max(),
            broadcast= pl.col(f"att_broadcast_{percentile}").mean(),
        )
        .sort(f"broadcast", "latency_bucket", "percentage", descending=True)
    )
    
    g = px.bar(
        df,
        x="latency_bucket",
        y="percentage",
        color=f"broadcast_{percentile}_bucket_g",
        opacity=0.9,
    )
    g.update_layout(
        title=f"Histogram of attestations first time seen by propagation {percentile}",
        xaxis_title_text="Seconds",
        yaxis_title_text="% of Attestations",
        barmode='stack',
        width=1200,
        height=800,
    )
    g.update_xaxes(range=[0, 14])
    g.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
    g.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
    render_plot_fn(g, render_method, f"./images/agg/att_histogram_by_first_seen_and_propagation{percentile}_tot.png")
    
histogram_per_first_att_seen_and_broadcast_percentile(attestation_df, "p50")
histogram_per_first_att_seen_and_broadcast_percentile(attestation_df, "p90")

## Attestation first seen vs propagation spread

Stacked histogram where the horizontal axis shows when each attestation was first observed by any control node and color axis shows the corresponding network-wide propagation bucket (p50 or p90). Identifies attestations that appeared early to one node but spread slowly.

In [ ]:
# CDF of the whole broadcast duration (P50 and P90) based on the time that they were seen
import plotly.io as pio
from IPython.display import Image

df_2 = (
    attestation_df
    .with_columns(
        total_attestations = pl.lit(len(attestation_df)),
    )
    .group_by(["latency_bucket", "inclusion_range"])
    .agg(
        attestations = pl.col("slot").count(),
        percentage = pl.col("slot").count() * 100 / pl.col("total_attestations").max(),
        inclusion= pl.col("inclusion_delay").mean(),
    )
    .sort("inclusion", "latency_bucket", "percentage", descending=True)
)

h_3 = px.bar(
    df_2,
    x="latency_bucket",
    y="percentage",
    color="inclusion_range",
)
h_3.update_layout(
    title="Histogram of when attestations where seen for the first time",
    xaxis_title_text="Seconds since slot started",
    yaxis_title_text="% of Attestations",
    barmode='stack',
    width=1200,
    height=800,
)
h_3.update_xaxes(range=[0, 12])
h_3.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
h_3.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
render_plot_fn(h_3, render_method, f"./images/agg/att_propagation_cdf_tot.png")

# Aggregations
Aggregate attestations are signed by a randomly selected aggregator from each committee and carry a bitfield covering all attesting validators in that committee. They are expected on the network from around second 8 of each slot, when aggregators publish. The charts below examine their first-seen timing, network propagation speed, and relationship to block arrival.

## Aggregation first seen

Histogram of when aggregate attestations were first observed by any control node, measured in seconds from slot start. The bulk of aggregations should appear around 8–9s.

In [ ]:
# Aggregations display first time seen (assuming that we have the timings for each messsage_id)
df = (
    aggregations_df
    .with_columns(
        total_aggregations = pl.lit(len(aggregations_df)),
    )
    .group_by(["latency_bucket"])
    .agg(
        percentage = pl.col("slot").count() * 100 / pl.col("total_aggregations").max(),
    )
    .sort("latency_bucket", descending=False)
)

g = px.bar(
    df,
    x="latency_bucket",
    y="percentage",
    opacity=0.9,
)
g.update_layout(
    title = "Histogram of first time seen the aggregations",
    xaxis_title_text="Seconds since slot start",
    yaxis_title_text="% of Aggregations",
    barmode='stack',
    width=1200,
    height=800,
)
g.update_xaxes(range=[7, 16])
render_plot_fn(g, render_method, "./images/agg/aggregation_histogram_first_seen_tot.png")


## Aggregation first seen by aggregate quality

Same histogram broken down by the number of bits set in the aggregation bitfield. Higher bit counts indicate larger, more useful aggregates. This reveals whether compact aggregates arrive earlier or later than their more complete counterparts.

In [ ]:
# Aggregations display first time seen (based on the number of aggregated bits that they have on the legend)

df = (
    aggregations_df
    .with_columns(
        total_aggregations = pl.lit(len(aggregations_df)),
    )
    .group_by(["latency_bucket", "aggregated_bits_bucket"])
    .agg(
        percentage = pl.col("slot").count() * 100 / pl.col("total_aggregations").max(),
    )
    .sort(["latency_bucket", "aggregated_bits_bucket"], descending=False)
)

g = px.bar(
    df,
    x="latency_bucket",
    y="percentage",
    color="aggregated_bits_bucket",
    opacity=0.9,
)
g.update_layout(
    title = "Histogram of first time seen the aggregations",
    xaxis_title_text="Seconds since slot start",
    yaxis_title_text="% of Aggregations",
    barmode='stack',
    width=1200,
    height=800,
)
g.update_xaxes(range=[7, 16])
render_plot_fn(g, render_method, "./images/agg/aggregation_histogram_first_seen_by_agg_bits_tot.png")


## Aggregation network propagation (p50 / p90)

Distribution of how long it took for each aggregation to be seen by 50% and 90% of control nodes, measured from slot start. Analogous to the attestation propagation charts but shifted to the 8–16s window. The gap between p50 and p90 indicates how evenly the aggregation gossip reaches the network.

In [ ]:
# Aggregations display propagation percentile (based on the number of aggregated bits that they have)
def arrival_of_aggregations_over_percentile(agg_df: pl.DataFrame, perc: str = 'p50'):
    df = (
        agg_df
        .with_columns(
            total_aggregations = pl.lit(len(agg_df)),
        )
        .group_by([f"broadcast_{perc}_bucket_wfs"])
        .agg(
            percentage = pl.col("slot").count() * 100 / pl.col("total_aggregations").max(),
            broadcast= pl.col(f"agg_broadcast_{perc}").mean(),
        )
        .sort(["broadcast"], descending=False)
    )
    
    # the histogram of when where the attestations seen
    g = px.bar(
        df,
        x=f"broadcast_{perc}_bucket_wfs",
        y="percentage",
    )
    g.update_layout(
        title=f"Histogram of when aggregations where seen by {perc} of control nodes",
        xaxis_title_text=f"{perc} propagation (s)",
        yaxis_title_text="% of Aggregations",
        width=1200,
        height=800,
    )
    g.update_xaxes(range=[7, 20])
    g.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
    g.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
    render_plot_fn(g, render_method, f"./images/agg/aggregation_histogram_{perc}_tot.png")
    
arrival_of_aggregations_over_percentile(aggregations_df, "p50")
arrival_of_aggregations_over_percentile(aggregations_df, "p90")

## Aggregation propagation by aggregate quality

P50/p90 propagation histogram broken down by aggregated bit-count. Helps identify whether high-quality aggregates (more bits set) propagate faster or slower than sparse ones, which could indicate prioritization effects in the gossip layer or correlation with specific aggregator clients.

In [ ]:
# Aggregations display propagation percentile (based on the number of aggregated bits that they have)
def arrival_of_aggregations_over_percentile(agg_df: pl.DataFrame, perc: str = 'p50'):
    df = (
        agg_df
        .with_columns(
            total_aggregations = pl.lit(len(agg_df)),
        )
        .group_by([f"broadcast_{perc}_bucket_wfs", "aggregated_bits_bucket"])
        .agg(
            percentage = pl.col("slot").count() * 100 / pl.col("total_aggregations").max(),
            broadcast= pl.col(f"agg_broadcast_{perc}").mean(),
        )
        .sort(["broadcast", "aggregated_bits_bucket"], descending=False)
    )
    
    # the histogram of when where the attestations seen
    g = px.bar(
        df,
        x=f"broadcast_{perc}_bucket_wfs",
        y="percentage",
        color="aggregated_bits_bucket",
        opacity=0.9,
    )
    g.update_layout(
        title=f"Histogram of when aggregations where seen by {perc} of control nodes",
        xaxis_title_text=f"{perc} propagation (s)",
        yaxis_title_text="% of Aggregations",
        barmode='stack',        
        width=1200,
        height=800,
    )
    g.update_xaxes(range=[7, 20])
    g.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
    g.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
    render_plot_fn(g, render_method, f"./images/agg/aggregation_histogram_{perc}_by_agg_bits_tot.png")
    
arrival_of_aggregations_over_percentile(aggregations_df, "p50")
arrival_of_aggregations_over_percentile(aggregations_df, "p90")

## Aggregation propagation duration

Raw propagation duration for each aggregation message, as the time from when the first node saw it until 50% or 90% of control nodes had received it. Unlike the slot-start-relative charts above, this measures within-network spread speed independent of when in the slot the aggregation was published.

In [ ]:
# Propagation CDF / Histogram
def plot_aggregation_propagation_time(agg_df: pl.DataFrame, perc: str = 'p50'):
    df = (
        agg_df
        .with_columns(
            total_aggregations = pl.lit(len(agg_df)),
        )
        .group_by([f"broadcast_{perc}_bucket"])
        .agg(
            percentage = pl.col("slot").count() * 100 / pl.col("total_aggregations").max(),
            broadcast= pl.col(f"agg_broadcast_{perc}").mean(),
        )
        .sort(["broadcast"], descending=False)
    )
    
    # the histogram of when where the attestations seen
    g = px.bar(
        df,
        x=f"broadcast_{perc}_bucket",
        y="percentage",
        opacity=0.9,
    )
    g.update_layout(
        title=f"Histogram the {perc} propagation of aggregations",
        xaxis_title_text=f"{perc} propagation (s)",
        yaxis_title_text="% of Aggregations",
        barmode='stack',        
        width=1200,
        height=800,
    )
    g.update_xaxes(range=[0, 8])
    render_plot_fn(g, render_method, f"./images/agg/aggregation_propagation_histogram_{perc}_tot.png")
    
plot_aggregation_propagation_time(aggregations_df, "p50")
plot_aggregation_propagation_time(aggregations_df, "p90")

## Aggregation propagation by slot

P50 propagation time for aggregations broken down by slot. Shows whether propagation quality is stable across slots or varies. E.g., slots with re-orgs, missed blocks, or heavy network load may exhibit different aggregation propagation patterns.

In [ ]:
def arrival_of_aggregations_over_percentile_and_slot(
    agg_df: pl.DataFrame,
    perc: str = 'p50',
):
    df = (
        agg_df
        .with_columns(
            total_aggregations = pl.lit(len(agg_df)),
        )
        .group_by([f"broadcast_{perc}_bucket_wfs", "slot"])
        .agg(
            percentage = pl.col("slot").count() * 100 / pl.col("total_aggregations").max(),
            broadcast= pl.col(f"agg_broadcast_{perc}").mean(),
        )
        .sort(["broadcast"], descending=False)
    )
    
    # the histogram of when where the attestations seen
    g = px.bar(
        df,
        x=f"broadcast_{perc}_bucket_wfs",
        y="percentage",
        color="slot",
        opacity=0.9,
    )
    g.update_layout(
        title=f"Histogram of when aggregations where seen by {perc} of control nodes",
        xaxis_title_text=f"{perc} propagation (s)",
        yaxis_title_text="% of Aggregations",
        barmode='stack',        
        width=1200,
        height=800,
    )
    g.update_xaxes(range=[7, 20])
    g.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
    g.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
    g.write_image(f"./images/agg/aggregation_histogram_{perc}__by_slot_tot.png")
    
arrival_of_aggregations_over_percentile_and_slot(aggregations_df, "p50")
arrival_of_aggregations_over_percentile_and_slot(aggregations_df, "p90")

## Aggregation first seen vs propagation spread

In [ ]:
def histogram_per_first_att_seen_and_broadcast_percentile(att_df: pl.DataFrame, perc: str):
    df = (
        att_df
        .with_columns(
            total = pl.lit(len(att_df)),
        )
        .group_by(["latency_bucket", f"broadcast_{perc}_bucket_g"])
        .agg(
            attestations = pl.col("slot").count(),
            percentage = pl.col("slot").count() * 100 / pl.col("total").max(),
            broadcast= pl.col(f"agg_broadcast_{perc}").mean(),
        )
        .sort(f"broadcast", "latency_bucket", "percentage", descending=True)
    )
    
    g = px.bar(
        df,
        x="latency_bucket",
        y="percentage",
        color=f"broadcast_{perc}_bucket_g",
        opacity=0.9,
    )
    g.update_layout(
        title=f"Histogram of aggregations first time seen by propagation {perc}",
        xaxis_title_text="Seconds since slot started",
        yaxis_title_text="% of Aggregations",
        barmode='stack',
        width=1200,
        height=800,
    )
    g.update_xaxes(range=[7, 13])
    g.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
    g.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
    render_plot_fn(g, render_method, f"./images/agg/agg_histogram_first_seen_and_{perc}.png")
    
histogram_per_first_att_seen_and_broadcast_percentile(aggregations_df, "p50")
histogram_per_first_att_seen_and_broadcast_percentile(aggregations_df, "p90")

## Aggregations per slot

Count of unique aggregation message IDs and unique aggregator indices observed per slot.

In [ ]:
# count of aggregations per slot

df = (
    aggregations_df
    .group_by(["slot"])
    .agg(
        unique_msgs=pl.col("message_id").count(),
        unique_aggregators=pl.col("aggregator_index").count(),
    )
    .sort(["slot"])
)

g = px.line(
    df.unpivot(index=["slot"], on=["unique_msgs", "unique_aggregators"], value_name="unique_items", variable_name="aggregations"),
    x="slot",
    y="unique_items",
    color="aggregations",
)
g.update_layout()
render_plot_fn(g, render_method, "./images/agg/unique_aggregators_and_msg_ids_per_slot.png")


## Full Slot Timeline

Blocks, data columns (PeerDAS), unaggregated attestations, and aggregate attestations are combined into a single timeline view spanning two consecutive slots. Block and column events from the next slot are shifted by +12s to show the full lifecycle: attestations propagate and get aggregated, then the next proposer publishes a block that includes them.

In [ ]:
block_flat_df = (
    block_and_column_df
    .select([ 
        "slot", 
        "block_first_seen", "block_latency_bucket",
        "block_broadcast_p50", "block_broadcast_p50_bucket", "block_broadcast_p50_bucket_wfs", "block_broadcast_p50_bucket_g",
        "block_broadcast_p90", "block_broadcast_p90_bucket", "block_broadcast_p90_bucket_wfs", "block_broadcast_p90_bucket_g",
    ])
    .with_columns(
        type=pl.lit("blocks"),
    )
    .rename({
        "block_first_seen": "first_seen",
        "block_latency_bucket": "latency_bucket",
        "block_broadcast_p50": "broadcast_p50",
        "block_broadcast_p50_bucket": "broadcast_p50_bucket",
        "block_broadcast_p50_bucket_wfs": "broadcast_p50_bucket_wfs",
        "block_broadcast_p50_bucket_g": "broadcast_p50_bucket_g",
        "block_broadcast_p90": "broadcast_p90",
        "block_broadcast_p90_bucket": "broadcast_p90_bucket",
        "block_broadcast_p90_bucket_wfs": "broadcast_p90_bucket_wfs",
        "block_broadcast_p90_bucket_g": "broadcast_p90_bucket_g",
    })
    .sort(["slot", "first_seen"])
    .unique()
)
block_flat_df = block_flat_df.with_columns(total=pl.lit(len(block_flat_df)))

columns_flat_df = (
    block_and_column_df
    .select([ 
        "slot", 
        "column_first_seen", "column_latency_bucket",
        "column_broadcast_p50", "column_broadcast_p50_bucket", "column_broadcast_p50_bucket_wfs", "column_broadcast_p50_bucket_g",
        "column_broadcast_p90", "column_broadcast_p90_bucket", "column_broadcast_p90_bucket_wfs", "column_broadcast_p90_bucket_g",
    ])
    .with_columns(
        type=pl.lit("columns"),
        total=pl.lit(len(aggregations_df)),
    )
    .rename({
        "column_first_seen": "first_seen",
        "column_latency_bucket": "latency_bucket",
        "column_broadcast_p50": "broadcast_p50",
        "column_broadcast_p50_bucket": "broadcast_p50_bucket",
        "column_broadcast_p50_bucket_wfs": "broadcast_p50_bucket_wfs",
        "column_broadcast_p50_bucket_g": "broadcast_p50_bucket_g",
        "column_broadcast_p90": "broadcast_p90",
        "column_broadcast_p90_bucket": "broadcast_p90_bucket",
        "column_broadcast_p90_bucket_wfs": "broadcast_p90_bucket_wfs",
        "column_broadcast_p90_bucket_g": "broadcast_p90_bucket_g",
    })
    .sort(["slot", "first_seen"])
    .unique()
)
columns_flat_df = columns_flat_df.with_columns(total=pl.lit(len(columns_flat_df)))

attestations_flat_df = (
    attestation_df
    .select([ 
        "slot", 
        "att_first_seen_wb", "latency_bucket",
        "att_broadcast_p50", "broadcast_p50_bucket", "broadcast_p50_bucket_wfs", "broadcast_p50_bucket_g",
        "att_broadcast_p90", "broadcast_p90_bucket", "broadcast_p90_bucket_wfs", "broadcast_p90_bucket_g",
    ])
    .rename({
        "att_first_seen_wb": "first_seen",
        "att_broadcast_p50": "broadcast_p50",
        "att_broadcast_p90": "broadcast_p90",
    })
    .with_columns(
        type=pl.lit("attestations"),
        first_seen=pl.col("first_seen").cast(pl.Float64),
        broadcast_p50=pl.col("broadcast_p50").cast(pl.Float64),
        broadcast_p90=pl.col("broadcast_p90").cast(pl.Float64),
        total=pl.lit(len(attestation_df)),
    )
    .sort(["slot", "first_seen"])
)


aggregations_flat_df = (
    aggregations_df
    .select([ 
        "slot", 
        "agg_first_seen_wb", "latency_bucket",
        "agg_broadcast_p50", "broadcast_p50_bucket", "broadcast_p50_bucket_wfs", "broadcast_p50_bucket_g",
        "agg_broadcast_p90", "broadcast_p90_bucket", "broadcast_p90_bucket_wfs", "broadcast_p90_bucket_g",
    ])
    .rename({
        "agg_first_seen_wb": "first_seen",
        "agg_broadcast_p50": "broadcast_p50",
        "agg_broadcast_p90": "broadcast_p90",
    })
    .with_columns(
        slot=pl.col("slot").cast(pl.UInt32),
        type=pl.lit("aggregations"),
        first_seen=pl.col("first_seen").cast(pl.Float64),
        broadcast_p50=pl.col("broadcast_p50").cast(pl.Float64),
        broadcast_p90=pl.col("broadcast_p90").cast(pl.Float64),
        total=pl.lit(len(aggregations_df)),
    )
    .sort(["slot", "first_seen"])
)

main_df = pl.concat([block_flat_df, columns_flat_df, attestations_flat_df, aggregations_flat_df]).sort("slot")
main_df = main_df.unique()


In [ ]:
# only show the aggregations + next block arrivals +  columns
next_block_flat_df = (
    block_and_column_df
    .select([ 
        "previous_slot",
        "block_first_seen", "block_latency_bucket",
        "block_broadcast_p50", "block_broadcast_p50_bucket", "block_broadcast_p50_bucket_wfs", "block_broadcast_p50_bucket_g",
        "block_broadcast_p90", "block_broadcast_p90_bucket", "block_broadcast_p90_bucket_wfs", "block_broadcast_p90_bucket_g",
    ])
    .with_columns(
        type=pl.lit("next_blocks")
    )
    .rename({
        "previous_slot": "slot",
        "block_first_seen": "first_seen",
        "block_latency_bucket": "latency_bucket",
        "block_broadcast_p50": "broadcast_p50",
        "block_broadcast_p50_bucket": "broadcast_p50_bucket",
        "block_broadcast_p50_bucket_wfs": "broadcast_p50_bucket_wfs",
        "block_broadcast_p50_bucket_g": "broadcast_p50_bucket_g",
        "block_broadcast_p90": "broadcast_p90",
        "block_broadcast_p90_bucket": "broadcast_p90_bucket",
        "block_broadcast_p90_bucket_wfs": "broadcast_p90_bucket_wfs",
        "block_broadcast_p90_bucket_g": "broadcast_p90_bucket_g",
    })
    .with_columns(
        slot=pl.col("slot").cast(pl.UInt32),
        first_seen=pl.col("first_seen")+12.0,
        latency_bucket=pl.col("latency_bucket")+12.0,
        broadcast_p50_bucket_wfs=pl.col("broadcast_p50_bucket_wfs")+12.0,
        broadcast_p90_bucket_wfs=pl.col("broadcast_p90_bucket_wfs")+12.0,
        total=pl.lit(len(aggregations_flat_df)),
    )
    .sort(["slot", "first_seen"])
)
next_block_flat_df = next_block_flat_df.with_columns(total=pl.lit(len(block_flat_df)))


next_columns_flat_df = (
    block_and_column_df
    .select([ 
        "slot", 
        "column_first_seen", "column_latency_bucket",
        "column_broadcast_p50", "column_broadcast_p50_bucket", "column_broadcast_p50_bucket_wfs", "column_broadcast_p50_bucket_g",
        "column_broadcast_p90", "column_broadcast_p90_bucket", "column_broadcast_p90_bucket_wfs", "column_broadcast_p90_bucket_g",
    ])
    .with_columns(
        type=pl.lit("next_columns"),
        total=pl.lit(len(aggregations_df)),
    )
    .rename({
        "column_first_seen": "first_seen",
        "column_latency_bucket": "latency_bucket",
        "column_broadcast_p50": "broadcast_p50",
        "column_broadcast_p50_bucket": "broadcast_p50_bucket",
        "column_broadcast_p50_bucket_wfs": "broadcast_p50_bucket_wfs",
        "column_broadcast_p50_bucket_g": "broadcast_p50_bucket_g",
        "column_broadcast_p90": "broadcast_p90",
        "column_broadcast_p90_bucket": "broadcast_p90_bucket",
        "column_broadcast_p90_bucket_wfs": "broadcast_p90_bucket_wfs",
        "column_broadcast_p90_bucket_g": "broadcast_p90_bucket_g",
    })
    .with_columns(
        slot=pl.col("slot").cast(pl.UInt32),
        first_seen=pl.col("first_seen")+12.0,
        latency_bucket=pl.col("latency_bucket")+12.0,
        broadcast_p50_bucket_wfs=pl.col("broadcast_p50_bucket_wfs")+12.0,
        broadcast_p90_bucket_wfs=pl.col("broadcast_p90_bucket_wfs")+12.0,
        total=pl.lit(len(aggregations_flat_df)),
    )
    .sort(["slot", "first_seen"])
    .unique()
)
next_columns_flat_df = next_columns_flat_df.with_columns(total=pl.lit(len(next_columns_flat_df)))

main_df_2 = pl.concat([next_block_flat_df, next_columns_flat_df, aggregations_flat_df]).unique().sort("slot")

## Aggregations vs next block and column arrival

Histogram comparing when aggregations were seen (first-seen for blocks/columns, p50 for aggregations) to when the next slot's block and data columns first arrived.

In [ ]:
# display aggregations with next blocks
# we want to visualize the p50 aggregations with the following slot arrival (first seen)
block_aux_df = (
    next_block_flat_df
    .select(["slot", "latency_bucket", "first_seen", "type", "total"])
    .rename({
        "latency_bucket": "broadcast_p50_bucket_wfs",
        "first_seen": "broadcast_p50",
    })
    .unique()
)

columns_aux_df = (
    next_columns_flat_df
    .select(["slot", "latency_bucket", "first_seen", "type", "total"])
    .rename({
        "latency_bucket": "broadcast_p50_bucket_wfs",
        "first_seen": "broadcast_p50",
    })
    .unique()
)

aggregations_aux_df = (
    aggregations_flat_df
    .select(["slot", "broadcast_p50_bucket_wfs", "broadcast_p50", "type", "total"])
)

df = (
    pl.concat([block_aux_df, aggregations_aux_df, columns_aux_df])
    .group_by(["type", "broadcast_p50_bucket_wfs"])
    .agg(
        percentage = pl.col("slot").count() * 100 / pl.col("total").max(),
        broadcast= pl.col(f"broadcast_p50").mean(),
    )
    .sort(["type", "broadcast"], descending=False)
)

# the histogram of when where the attestations seen
g = px.bar(
    df,
    x=f"broadcast_p50_bucket_wfs",
    y="percentage",
    color="type",
)
g.update_layout(
    title=f"Histogram of when aggregations were seen by p50 of control nodes and when the next block and columns were first seen",
    xaxis_title_text=f"Seconds since the slot started",
    yaxis_title_text="% of messages",
    barmode='group',
    width=1200,
    height=800,
)
g.update_xaxes(range=[0, 20])
g.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
g.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
render_plot_fn(g, render_method, "./images/agg/aggregation_p50_and_next_block_correlation_histogram_tot.png")


## Aggregations vs next slot (p50 network perspective)

Same cross-slot comparison using p50 propagation time for blocks and columns instead of first-seen. Shows whether the network-wide spread of aggregations aligns with the network-wide arrival of the following block.

In [ ]:
# we want to visualize the p50 aggregations with the following slot arrival's p50
df = (
    main_df_2
    .group_by([f"broadcast_p50_bucket_wfs", "type"])
    .agg(
        percentage = pl.col("slot").count() * 100 / pl.col("total").max(),
        broadcast= pl.col(f"broadcast_p50").mean(),
    )
    .sort(["type", "broadcast"], descending=False)
)

# the histogram of when where the attestations seen
g = px.bar(
    df,
    x=f"broadcast_p50_bucket_wfs",
    y="percentage",
    color="type",
)
g.update_layout(
    title=f"Histogram of when aggregationsa and the next block + columns were seen by p50 of control nodes",
    xaxis_title_text=f"Seconds since the slot started",
    yaxis_title_text="% of messages",
    barmode='group',
    width=1200,
    height=800,
)
g.update_xaxes(range=[0, 20])
g.add_vline(x=8, line_width=3, line_dash="dash", line_color="purple")
g.add_vline(x=12, line_width=3, line_dash="dash", line_color="red")
render_plot_fn(g, render_method, "./images/agg/aggregation_p50_and_next_block_correlation_histogram_p50_tot.png")